# GEANT h96 reproducibility matrix

Preparation and preflight are automated; the last cell is a separate, disabled-by-default training gate. The new matrix source is a review candidate: its digest below guarantees bytes, not approval. Independently review and approve it before changing `SOURCE_REVIEW_APPROVED`. The dataset remains the approved train/validation-only package. The official test split is not packaged or accessed.

Operator: select a CPU Colab runtime, run cells top-to-bottom, upload the four approved bundle files when prompted (names may have `(1)` suffixes; hashes decide identity), inspect PASS results and the dry-run, then stop for source review. Only after separate review/approval should you enable the final cell. It asks for one overall confirmation and the launcher asks for an exact separate phrase before each remaining run. Seed-42/topology is excluded. Colab execution itself has not been independently observed here.

In [ ]:
# Pinned runtime; preparation only.
import sys, subprocess, importlib.metadata as ilm
if sys.version_info[:2] != (3,12): raise RuntimeError(f'Requires Python 3.12; found {sys.version.split()[0]}')
PINS={'numpy':'1.26.4','torch':'2.12.0','psutil':'7.2.2','pytest':'9.1.1'}
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*[f'{k}=={v}' for k,v in PINS.items()]],check=True)
actual={k:ilm.version(k) for k in PINS}
if actual!=PINS: raise RuntimeError(f'Package pins mismatch: {actual}; restart runtime and rerun all cells')
import torch
if torch.cuda.is_available(): raise RuntimeError('Select CPU runtime; CUDA is not part of this protocol')
print('PASS: Python/packages/CPU',sys.version.split()[0],actual)

In [ ]:
# Upload, identify all files by pinned SHA-256, verify full ZIP inventories/member hashes, and extract only to new paths.
import hashlib, importlib, json, os, sys, uuid, zipfile
from pathlib import Path
from google.colab import files
EXPECTED={'geant_h96_colab_source_v1.zip':'b5d484edd60416185c57fd1c3f4de450b97b3b70ef9a453152cb86aca5ecc574','source_snapshot_manifest.json':'c2661323866d2089a0f06f89158fcc46cbd5c0022088b26262a9cfd8b08dd8be','geant_h96_colab_trainval_v1.zip':'df2bac044c9b9a577d5117310a99022cce402b9370e2ba483ea18b8d7045521e','dataset_bundle_manifest.json':'70f60f09b71c0cb89003ac2987af4433df0eeeb64b696845faa25d0fbadb8a95'}
def digest(b): return hashlib.sha256(b).hexdigest()
uploaded=files.upload()
if len(uploaded)!=4: raise RuntimeError(f'Upload exactly four bundle files; received {len(uploaded)}')
byhash={}
for display,body in uploaded.items(): byhash.setdefault(digest(body),[]).append(body)
resolved={}
for name,h in EXPECTED.items():
    if len(byhash.get(h,[]))!=1: raise RuntimeError(f'Expected one file by exact hash for {name}; duplicate/missing/mismatched upload')
    resolved[name]=byhash[h][0]
if sum(len(v) for v in byhash.values())!=4: raise RuntimeError('Extra or duplicate upload refused')
SESSION_ID=uuid.uuid4().hex; SESSION=Path('/content')/f'geant_h96_matrix_v1_{SESSION_ID}'; SESSION.mkdir(exist_ok=False)
INCOMING=SESSION/'incoming'; INCOMING.mkdir(exist_ok=False)
for name,body in resolved.items():
    with (INCOMING/name).open('xb') as f: f.write(body); f.flush(); os.fsync(f.fileno())
# Import package code from the source ZIP only after its independently pinned whole-archive digest matches.
source_zip=INCOMING/'geant_h96_colab_source_v1.zip'; source_manifest=INCOMING/'source_snapshot_manifest.json'
sys.path.insert(0,str(source_zip))
pkg=importlib.import_module('phase9_geant.colab_h96_v1.package')
src_check=pkg.verify_source_bundle(source_zip,source_manifest,EXPECTED[source_zip.name],EXPECTED[source_manifest.name])
data_zip=INCOMING/'geant_h96_colab_trainval_v1.zip'; data_manifest=INCOMING/'dataset_bundle_manifest.json'
data_check=pkg.verify_dataset_bundle(data_zip,data_manifest,EXPECTED[data_zip.name],EXPECTED[data_manifest.name])
src_obj=json.loads(source_manifest.read_text()); REPO=SESSION/'approved_source_snapshot'
pkg.extract_source_bundle(source_zip,REPO,src_obj)
sys.path.remove(str(source_zip)); sys.path.insert(0,str(REPO))
pkg=pkg # verified extracted runtime copy
pkg.verify_extracted_source(REPO,src_obj)
pkg.extract_dataset_bundle(data_zip,REPO)
DATA_ROOT=REPO/pkg.DATA_ROOT_RELATIVE
if {p.name for p in DATA_ROOT.iterdir()}!=set(pkg.DATA_ARTIFACTS): raise RuntimeError('Exact dataset allowlist mismatch')
meta=pkg._read_dataset_metadata(DATA_ROOT)
if meta['windows']['train_count']!=3924 or meta['windows']['validation_count']!=953: raise RuntimeError('Window count mismatch')
from phase9_geant.colab_h96_v1 import environment, linux_resources, matrix_workflow
ENV=environment.verify_environment(require_pytest=True)
if linux_resources.RESOURCE_POLICY_ID!='colab-linux-process-memory-v3-visible-cgroup' or linux_resources.PROCESS_PEAK_LIMIT_BYTES!=2*1024**3: raise RuntimeError('Expected v3 2-GiB resource policy absent')
print('PASS: source/data archives verified',{'source_members':len(src_check['members']),'dataset_members':len(data_check['members']),'windows':meta['windows'],'test_members':0,'policy':linux_resources.RESOURCE_POLICY_ID})

In [ ]:
# Synthetic suite and matrix dry-run. Neither starts training.
import json, subprocess, sys, uuid
TESTS=['phase9_geant/colab_h96_v1/tests/test_portability.py','phase9_geant/colab_h96_v1/tests/test_launch_matrix.py','phase9_geant/colab_h96_v1/tests/test_matrix_workflow.py','phase9_geant/tests/test_horizon96_streaming_experiment.py','phase9_geant/tests/test_horizon96_streaming_experiment_v2.py','phase9_geant/tests/test_spatiotemporal_link.py']
BASE=SESSION/f'pytest_{uuid.uuid4().hex}'
test=subprocess.run([sys.executable,'-m','pytest','-q','-rs','--basetemp',str(BASE),*TESTS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode!=0 or 'failed' in test.stdout.lower(): raise RuntimeError(f'Synthetic suite failed, exit {test.returncode}')
ARGS=['--repository-root',str(REPO),'--source-bundle',str(INCOMING/'geant_h96_colab_source_v1.zip'),'--source-manifest',str(INCOMING/'source_snapshot_manifest.json'),'--expected-source-bundle-sha256',EXPECTED['geant_h96_colab_source_v1.zip'],'--expected-source-manifest-sha256',EXPECTED['source_snapshot_manifest.json'],'--dataset-bundle',str(INCOMING/'geant_h96_colab_trainval_v1.zip'),'--dataset-manifest',str(INCOMING/'dataset_bundle_manifest.json'),'--expected-dataset-bundle-sha256',EXPECTED['geant_h96_colab_trainval_v1.zip'],'--expected-dataset-manifest-sha256',EXPECTED['dataset_bundle_manifest.json']]
LAUNCH=REPO/'phase9_geant/colab_h96_v1/launch_matrix.py'
dry=subprocess.run([sys.executable,'-I','-S',str(LAUNCH),'--dry-run',*ARGS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(dry.stdout)
if dry.returncode!=0: raise RuntimeError(f'Matrix dry-run failed, exit {dry.returncode}')
DRY=json.loads(dry.stdout)
if DRY.get('status')!='preflight_passed_no_training' or DRY.get('resource_policy',{}).get('policy_id')!='colab-linux-process-memory-v3-visible-cgroup': raise RuntimeError('Dry-run status/policy mismatch')
PREFLIGHT={'status':DRY['status'],'matrix_id':DRY['matrix_id'],'protocol_id':DRY['protocol_id'],'output_root':DRY['output_root'],'planned_runs':DRY['planned_runs'],'source_sha256':EXPECTED['geant_h96_colab_source_v1.zip'],'dataset_sha256':EXPECTED['geant_h96_colab_trainval_v1.zip'],'environment':ENV,'resource_policy':DRY['resource_policy'],'tests_output':test.stdout,'test_split_accessed':False}
(SESSION/'preflight.json').write_text(json.dumps(PREFLIGHT,indent=2),encoding='utf-8')
print('PASS: preflight_passed_no_training; no run launched')

## Separate training authorization

The following cell does nothing unless the source bundle has been separately reviewed/approved and you explicitly set the two controls. The launcher then obtains a unique exact confirmation before each of the eight runs. It never includes completed seed-42/topology. Keep this cell unexecuted until approval. A failed attempt is preserved and must not be retried in this matrix namespace.

In [ ]:
# Explicitly disabled defaults. Training is not performed by notebook preparation.
SOURCE_REVIEW_APPROVED=False
RUN_MATRIX_AUTHORIZED=False
if not SOURCE_REVIEW_APPROVED: raise PermissionError('Blocked: independently review and approve the new matrix source bundle first')
if PREFLIGHT.get('status')!='preflight_passed_no_training' or PREFLIGHT.get('test_split_accessed') is not False: raise RuntimeError('No valid current-session preflight receipt')
if input('Type AUTHORIZE H96 MATRIX to begin the eight remaining runs: ').strip()!='AUTHORIZE H96 MATRIX': raise PermissionError('No authorization; nothing launched')
RUN_MATRIX_AUTHORIZED=True
command=[sys.executable,'-I','-S',str(LAUNCH),'--run-matrix',*ARGS]
log=SESSION/'matrix_launcher.log'
with log.open('x',encoding='utf-8') as stream:
    def emit(line): print(line,flush=True); stream.write(line+'\n'); stream.flush(); os.fsync(stream.fileno())
    def authorize_one(phrase): return input('Type exactly '+phrase+':\n')
    code=None
    try: code=matrix_workflow.run_matrix_with_prompt_bridge(command,prompt_fn=authorize_one,output_fn=emit)
    finally:
        root=REPO/'data/processed/geant_horizon96_streaming_experiment_v3_repro_matrix1'
        if root.exists() and (root/'matrix_run.json').is_file():
            export=SESSION/f'matrix_export_{SESSION_ID}.zip'
            matrix_workflow.export_matrix_results(root,export)
            print('Export created:',export)
            files.download(str(export))
    if code!=0: raise RuntimeError(f'Matrix stopped with exit {code}; preserve partial artifacts and do not retry')
print('Matrix launcher completed. Review matrix_run.json and each run artifact before interpreting results.')